In [19]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import LabelEncoder

## Load and Merge Datasets
in this section we load both our csv files and merge them into one dataframe

In [3]:
train_id = pd.read_csv("../data/raw/train_identity.csv")
train_trans = pd.read_csv("../data/raw/train_transaction.csv")

df = pd.merge(left=train_trans, right=train_id, how="left",  on="TransactionID")

In [4]:
df.shape

(590540, 434)

## **R_emaildomain -- Frequency based Grouping**
we assign numeric values 1-5 for the top 5 email provider used in our dataset. As we do have many nullvalues (80%) and have other uncommon domains, we group those together under the numeric value 6

In [6]:
top5_email_categories = df["R_emaildomain"].value_counts().head(5)

In [7]:
email_mapper = {}
for i, domain in enumerate(top5_email_categories.index.tolist()):
    email_mapper[domain] = i+1

In [8]:
df["R_emaildomain"] = df["R_emaildomain"].map(email_mapper).fillna(6).astype(int)

## **One-Hot-Encoding**
in this section we take multiple string columns with only a few (<10) different values and apply one hot encoding

### ProductCD Column

In [9]:
df = pd.get_dummies(data=df, prefix="ProductCD", columns=["ProductCD"])

### card4 Column

In [10]:
df = pd.get_dummies(data=df, columns=["card4"], prefix="card4")

### card6 Column

In [11]:
df = pd.get_dummies(data=df, prefix="card6", columns=["card6"])

### DeviceType column

In [12]:
df = pd.get_dummies(data=df, prefix="DeviceType", columns=["DeviceType"])

## Time Column 
as we found out in our EDA, the "TransactionDT" corresponds to the unit second.
Therefore we can create a weekday (0-6) and a hour (0-23) column.
This allows us as humans to properly interpret the data, while also enabling the Modell to find patterns more easily

In [13]:
df["DT_day"] = df["TransactionDT"] // 86400
df["DT_hour"] = (df["TransactionDT"] // 3600) % 24
df["DT_weekday"] = df["DT_day"] % 7

df = df.drop(["DT_day", "TransactionDT"], axis=1)

/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_59204/3280605987.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_day"] = df["TransactionDT"] // 86400
/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_59204/3280605987.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_hour"] = (df["TransactionDT"] // 3600) % 24
/var/folders/4m/khzpjqdj6cl07kznw50hb5780000gn/T/ipykernel_59204/3280605987.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

## Label Encoder
we want to make sure that NaN values are being natively handled by XGBoost. Therefore we want to remain those nan values during LabelEncoding. To ensure NaN values are being kept, we mask only the non NaN values and apply the transformation exclusively for those. We do so by creating a Series, mapping via boolean if a given Value is a string or nan(string = true, Nan = false). Then we create a new, NaN filled column (encoded). All index of mask in encoded now being transformed by our label encoder to become numeric values.

In [ ]:
le = LabelEncoder() # init LabelEncoder

In [ ]:

cat_cols = df.select_dtypes(include="str").columns # select all columns with string instead of numeric values

for col in cat_cols: # loop over each string column
    mask = df[col].notna() # Series of boolean values (for NaN = false, rest = True)
    encoded = pd.Series(index=df.index, dtype="float64") # create new, empty series requiring float as type with the number of rows (df.index). Base Value = NaN
    encoded[mask] = le.fit_transform(df.loc[mask, col]) #encoded[mask]= only change index values of mask, leave rest.  df.loc[mask, col] = only take rows where mask = True, only take this column
    df[col] = encoded